In [2]:
import os
import sys
import json

from pathlib import Path
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

BASE = Path.home() / "BigData_UPAO" / "proyecto_afp_v2"
EVIDENCIAS = BASE / "evidencias"

EVIDENCIAS.mkdir(parents=True, exist_ok=True)

HDFS_BASE = (
    "hdfs://localhost:9000"
    "/user/junior/datalake_afp_v2"
)

derby_path = (
    BASE / "metastore" / "metastore_db"
).as_posix()

os.environ["HADOOP_HOME"] = "/opt/hadoop-3.3.6"
os.environ["HADOOP_CONF_DIR"] = (
    "/opt/hadoop-3.3.6/etc/hadoop"
)
os.environ["YARN_CONF_DIR"] = os.environ["HADOOP_CONF_DIR"]

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "10.0.2.15"

spark = (
    SparkSession.builder
    .appName("AFP_V2_Benchmark_Rendimiento")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.driver.host", "10.0.2.15")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.ui.port", "4052")
    .config("spark.executor.instances", "1")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "512m")
    .config(
        "spark.hadoop.javax.jdo.option.ConnectionURL",
        f"jdbc:derby:;databaseName={derby_path};create=true"
    )
    .config(
        "spark.sql.warehouse.dir",
        HDFS_BASE + "/catalog/warehouse"
    )
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("=== CONEXIÓN SPARK ===")

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print(
    "Catálogo:",
    spark.conf.get("spark.sql.catalogImplementation")
)

print("\n=== BASES DE DATOS ===")

spark.sql("SHOW DATABASES").show()

=== CONEXIÓN SPARK ===
Spark: 3.5.1
Master: yarn
Catálogo: hive

=== BASES DE DATOS ===
+-------------+
|    namespace|
+-------------+
|afp_analytics|
|      default|
+-------------+



In [3]:
import time
import json
import statistics
import subprocess
from pathlib import Path
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

BASE = Path.home() / "BigData_UPAO" / "proyecto_afp_v2"
EVIDENCIAS = BASE / "evidencias"
EVIDENCIAS.mkdir(parents=True, exist_ok=True)

print("=== ENTORNO DEL BENCHMARK ===")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Catálogo:", spark.conf.get("spark.sql.catalogImplementation"))
print("Paralelismo por defecto:", spark.sparkContext.defaultParallelism)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

print("\n=== RECURSOS DE LA MAQUINA ===")

subprocess.run(["free", "-h"], check=True)

print("\n=== TABLAS ORIGINALES ===")

tablas = [
    "bcrp_series_largas",
    "bcrp_json_rentabilidad",
    "sbs_afiliados",
    "sbs_rentabilidad",
    "sbs_valor_cuota",
    "memorias_indicadores",
    "sbs_indicadores_adicionales",
    "tabla_analitica_mensual"
]

conteos = {
    nombre: spark.table(f"afp_analytics.{nombre}").count()
    for nombre in tablas
}

print("Tablas:", len(conteos))
print("Filas acumuladas:", sum(conteos.values()))

assert sum(conteos.values()) == 110407

=== ENTORNO DEL BENCHMARK ===
Spark: 3.5.1
Master: yarn
Catálogo: hive
Paralelismo por defecto: 2
Shuffle partitions: 200

=== RECURSOS DE LA MAQUINA ===
               total        used        free      shared  buff/cache   available
Mem:           7.8Gi       5.7Gi       148Mi        56Mi       2.2Gi       2.0Gi
Swap:          974Mi        27Mi       947Mi

=== TABLAS ORIGINALES ===


Tablas: 8
Filas acumuladas: 110407


In [5]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.adaptive.enabled", "false")

cuotas = (
    spark.table("afp_analytics.sbs_valor_cuota")
    .select(
        "fecha",
        "afp",
        "fondo",
        "valor_cuota",
        "fecha_mes"
    )
)

mensual = (
    spark.table("afp_analytics.tabla_analitica_mensual")
    .select(
        "fecha_mes",
        "rent_real_integra_12m"
    )
)

print("=== TAMAÑO DE LAS FUENTES ===")
print("Valor cuota:", cuotas.count())
print("Tabla mensual:", mensual.count())

print("\n=== ESQUEMAS ===")
cuotas.printSchema()
mensual.printSchema()

=== TAMAÑO DE LAS FUENTES ===


Valor cuota: 76059
Tabla mensual: 140

=== ESQUEMAS ===
root
 |-- fecha: date (nullable = true)
 |-- afp: string (nullable = true)
 |-- fondo: integer (nullable = true)
 |-- valor_cuota: double (nullable = true)
 |-- fecha_mes: date (nullable = true)

root
 |-- fecha_mes: date (nullable = true)
 |-- rent_real_integra_12m: double (nullable = true)



In [6]:
def consulta_sin_broadcast():

    return (
        cuotas.join(
            mensual,
            on="fecha_mes",
            how="inner"
        )
        .withColumn(
            "valor_ponderado",
            F.col("valor_cuota")
            * F.col("rent_real_integra_12m")
        )
        .groupBy("afp", "fondo")
        .agg(
            F.count("*").alias("registros"),
            F.sum("valor_ponderado").alias("total_ponderado")
        )
    )


def consulta_con_broadcast():

    return (
        cuotas.join(
            broadcast(mensual),
            on="fecha_mes",
            how="inner"
        )
        .withColumn(
            "valor_ponderado",
            F.col("valor_cuota")
            * F.col("rent_real_integra_12m")
        )
        .groupBy("afp", "fondo")
        .agg(
            F.count("*").alias("registros"),
            F.sum("valor_ponderado").alias("total_ponderado")
        )
    )

In [7]:
print("=== ANTES: UNION SIN BROADCAST ===")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

consulta_sin_broadcast().explain(True)

print("\n=== DESPUES: UNION CON BROADCAST ===")

consulta_con_broadcast().explain(True)

=== ANTES: UNION SIN BROADCAST ===
== Parsed Logical Plan ==
'Aggregate ['afp, 'fondo], ['afp, 'fondo, count(1) AS registros#595L, sum('valor_ponderado) AS total_ponderado#597]
+- Project [fecha_mes#109, fecha#105, afp#107, fondo#106, valor_cuota#108, rent_real_integra_12m#264, (valor_cuota#108 * rent_real_integra_12m#264) AS valor_ponderado#579]
   +- Project [fecha_mes#109, fecha#105, afp#107, fondo#106, valor_cuota#108, rent_real_integra_12m#264]
      +- Join Inner, (fecha_mes#109 = fecha_mes#261)
         :- Project [fecha#105, afp#107, fondo#106, valor_cuota#108, fecha_mes#109]
         :  +- SubqueryAlias spark_catalog.afp_analytics.sbs_valor_cuota
         :     +- Relation spark_catalog.afp_analytics.sbs_valor_cuota[fecha#105,fondo#106,afp#107,valor_cuota#108,fecha_mes#109,anio#110] parquet
         +- Project [fecha_mes#261, rent_real_integra_12m#264]
            +- SubqueryAlias spark_catalog.afp_analytics.tabla_analitica_mensual
               +- Relation spark_catalog.afp_

In [8]:
def ejecutar_y_resumir(construir):

    df = construir()

    filas = df.orderBy("afp", "fondo").collect()

    return [
        (
            r["afp"],
            r["fondo"],
            r["registros"],
            r["total_ponderado"]
        )
        for r in filas
    ]


def medir(nombre, construir, repeticiones=3):

    print("\nPROBANDO:", nombre)

    # Calentamiento
    referencia = ejecutar_y_resumir(construir)

    tiempos = []

    for i in range(repeticiones):

        inicio = time.perf_counter()

        resultado = ejecutar_y_resumir(construir)

        segundos = time.perf_counter() - inicio

        assert len(resultado) == len(referencia)

        tiempos.append(segundos)

        print(
            f"Repetición {i + 1}: "
            f"{segundos:.3f} segundos"
        )

    return {
        "estrategia": nombre,
        "tiempos_segundos": tiempos,
        "mediana_segundos": statistics.median(tiempos),
        "resultado": referencia
    }


spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

antes = medir(
    "Shuffle join sin broadcast",
    consulta_sin_broadcast
)

despues = medir(
    "Broadcast join",
    consulta_con_broadcast
)

print("\n=== COMPARACIÓN FINAL ===")

print(
    "Tiempo antes:",
    round(antes["mediana_segundos"], 3),
    "segundos"
)

print(
    "Tiempo después:",
    round(despues["mediana_segundos"], 3),
    "segundos"
)

mejora_pct = (
    (antes["mediana_segundos"] - despues["mediana_segundos"])
    / antes["mediana_segundos"] * 100
)

print("Variación porcentual:", round(mejora_pct, 2), "%")


PROBANDO: Shuffle join sin broadcast


Repetición 1: 2.001 segundos
Repetición 2: 1.554 segundos


Repetición 3: 1.651 segundos

PROBANDO: Broadcast join


Repetición 1: 1.238 segundos
Repetición 2: 1.295 segundos
Repetición 3: 1.445 segundos

=== COMPARACIÓN FINAL ===
Tiempo antes: 1.651 segundos
Tiempo después: 1.295 segundos
Variación porcentual: 21.52 %


In [10]:
import math

resultado_antes = sorted(
    antes["resultado"],
    key=lambda x: (str(x[0]), str(x[1]))
)

resultado_despues = sorted(
    despues["resultado"],
    key=lambda x: (str(x[0]), str(x[1]))
)

assert len(resultado_antes) == len(resultado_despues)

for a, b in zip(resultado_antes, resultado_despues):

    assert a[:3] == b[:3]

    assert math.isclose(
        a[3] or 0.0,
        b[3] or 0.0,
        rel_tol=1e-9,
        abs_tol=1e-6
    )

print("RESULTADOS EQUIVALENTES: SI")
print("Grupos comparados:", len(resultado_antes))

RESULTADOS EQUIVALENTES: SI
Grupos comparados: 16


In [11]:
reporte_optimizacion = {
    "fecha": datetime.now().isoformat(),
    "spark": spark.version,
    "master": spark.sparkContext.master,
    "tecnica": "broadcast join",
    "tiempos_antes_segundos": antes["tiempos_segundos"],
    "tiempos_despues_segundos": despues["tiempos_segundos"],
    "mediana_antes_segundos": antes["mediana_segundos"],
    "mediana_despues_segundos": despues["mediana_segundos"],
    "variacion_pct": mejora_pct,
    "resultados_equivalentes": True
}

ruta = EVIDENCIAS / "17_optimizacion_broadcast.json"

with ruta.open("w", encoding="utf-8") as f:
    json.dump(reporte_optimizacion, f, indent=2)

print("Evidencia:", ruta)

Evidencia: /home/junior/BigData_UPAO/proyecto_afp_v2/evidencias/17_optimizacion_broadcast.json


In [12]:
print("=== CONFIGURACIÓN DE ESCALABILIDAD ===")

ESCALAS = [1, 10, 100]

FILAS_BASE = sum(conteos.values())

RUTA_BENCHMARK = (
    "hdfs://localhost:9000"
    "/user/junior/datalake_afp_v2/benchmark"
)

for escala in ESCALAS:
    print(
        f"Escala {escala}x:",
        f"{FILAS_BASE * escala:,} filas de referencia"
    )

print("\nRuta experimental:", RUTA_BENCHMARK)

print("\nSpark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Paralelismo:", spark.sparkContext.defaultParallelism)

assert FILAS_BASE == 110407

=== CONFIGURACIÓN DE ESCALABILIDAD ===
Escala 1x: 110,407 filas de referencia
Escala 10x: 1,104,070 filas de referencia
Escala 100x: 11,040,700 filas de referencia

Ruta experimental: hdfs://localhost:9000/user/junior/datalake_afp_v2/benchmark

Spark: 3.5.1
Master: yarn
Paralelismo: 2


In [14]:
import subprocess

print("=== MEMORIA DISPONIBLE ===")
subprocess.run(["free", "-h"], check=True)

print("\n=== ESPACIO LOCAL ===")
subprocess.run(["df", "-h", "/"], check=True)

print("\n=== CAPACIDAD DE HDFS ===")
subprocess.run(["hdfs", "dfsadmin", "-report"], check=True)

print("\n=== NODOS DE YARN ===")
subprocess.run(["yarn", "node", "-list"], check=True)

print("\n=== APLICACIONES ACTIVAS ===")
subprocess.run(["yarn", "application", "-list"], check=True)

=== MEMORIA DISPONIBLE ===
               total        used        free      shared  buff/cache   available
Mem:           7.8Gi       6.1Gi       251Mi        56Mi       1.7Gi       1.6Gi
Swap:          974Mi        93Mi       881Mi

=== ESPACIO LOCAL ===

=== CAPACIDAD DE HDFS ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/sda1        39G   17G   20G  45% /
Configured Capacity: 40947412992 (38.14 GB)
Present Capacity: 22917259264 (21.34 GB)
DFS Remaining: 21380550656 (19.91 GB)
DFS Used: 1536708608 (1.43 GB)
DFS Used%: 6.71%
Replicated Blocks:
	Under replicated blocks: 726
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 726
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0


2026-10-11 10:31:54,856 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032


Total Nodes:1
         Node-Id	     Node-State	Node-Http-Address	Number-of-Running-Containers
vbox.myguest.virtualbox.org:46339	        RUNNING	vbox.myguest.virtualbox.org:8042	                           2

=== APLICACIONES ACTIVAS ===


2026-10-11 10:31:56,294 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032


Total number of applications (application-types: [], states: [SUBMITTED, ACCEPTED, RUNNING] and tags: []):1
                Application-Id	    Application-Name	    Application-Type	      User	     Queue	             State	       Final-State	       Progress	                       Tracking-URL
application_1791724084229_0002	AFP_V2_Benchmark_Rendimiento	               SPARK	    junior	   default	           RUNNING	         UNDEFINED	            10%	              http://10.0.2.15:4052


CompletedProcess(args=['yarn', 'application', '-list'], returncode=0)

In [15]:
from pyspark.sql import functions as F
from datetime import datetime
import time

TABLAS_BENCHMARK = [
    "bcrp_series_largas",
    "bcrp_json_rentabilidad",
    "sbs_afiliados",
    "sbs_rentabilidad",
    "sbs_valor_cuota",
    "memorias_indicadores",
    "sbs_indicadores_adicionales",
    "tabla_analitica_mensual"
]

ESCALA_PRUEBA = 1

ID_EXPERIMENTO = datetime.now().strftime("%Y%m%d_%H%M%S")

RUTA_EXPERIMENTO = (
    RUTA_BENCHMARK + "/" + ID_EXPERIMENTO
)

print("=== PRUEBA DE ESCALA 1X ===")
print("Identificador:", ID_EXPERIMENTO)
print("Destino:", RUTA_EXPERIMENTO)

def replicar_tabla(nombre, escala):
    original = spark.table(f"afp_analytics.{nombre}")

    copias = spark.range(escala).select(
        F.col("id").cast("int").alias("replica_id")
    )

    return original.crossJoin(F.broadcast(copias))

print("\n=== CONTEOS DE LA RÉPLICA ===")

for nombre in TABLAS_BENCHMARK:
    original = conteos[nombre]
    esperado = original * ESCALA_PRUEBA

    print(f"{nombre}: {esperado:,} filas esperadas")

print("\nTOTAL ESPERADO:", sum(conteos.values()))

=== PRUEBA DE ESCALA 1X ===
Identificador: 20261011_103456
Destino: hdfs://localhost:9000/user/junior/datalake_afp_v2/benchmark/20261011_103456

=== CONTEOS DE LA RÉPLICA ===
bcrp_series_largas: 12,317 filas esperadas
bcrp_json_rentabilidad: 699 filas esperadas
sbs_afiliados: 1,148 filas esperadas
sbs_rentabilidad: 2,898 filas esperadas
sbs_valor_cuota: 76,059 filas esperadas
memorias_indicadores: 48 filas esperadas
sbs_indicadores_adicionales: 17,098 filas esperadas
tabla_analitica_mensual: 140 filas esperadas

TOTAL ESPERADO: 110407


In [16]:
import psutil
import time

resultados_escala_1x = []

inicio_total = time.perf_counter()

print("=== EJECUCIÓN EXPERIMENTAL 1X ===")

for nombre in TABLAS_BENCHMARK:

    inicio = time.perf_counter()

    df = replicar_tabla(nombre, ESCALA_PRUEBA)

    registros = df.count()
    esperado = conteos[nombre] * ESCALA_PRUEBA

    assert registros == esperado, (
        f"Error de cantidad en {nombre}: "
        f"{registros} frente a {esperado}"
    )

    destino = (
        f"{RUTA_EXPERIMENTO}/escala_1x/"
        f"{nombre}"
    )

    (
        df.write
        .mode("errorifexists")
        .parquet(destino)
    )

    segundos = time.perf_counter() - inicio

    memoria = psutil.virtual_memory()

    resultados_escala_1x.append({
        "tabla": nombre,
        "filas": registros,
        "tiempo_segundos": round(segundos, 3),
        "ram_sistema_usada_pct": memoria.percent
    })

    print(
        f"{nombre}: {registros:,} filas | "
        f"{segundos:.3f} s | "
        f"RAM sistema: {memoria.percent:.1f}%"
    )

tiempo_total = time.perf_counter() - inicio_total

print("\n=== RESULTADO 1X ===")
print("Tablas procesadas:", len(resultados_escala_1x))
print(
    "Filas acumuladas:",
    sum(r["filas"] for r in resultados_escala_1x)
)
print("Tiempo total:", round(tiempo_total, 3), "segundos")

=== EJECUCIÓN EXPERIMENTAL 1X ===


bcrp_series_largas: 12,317 filas | 3.330 s | RAM sistema: 79.2%


bcrp_json_rentabilidad: 699 filas | 2.199 s | RAM sistema: 79.3%


sbs_afiliados: 1,148 filas | 1.555 s | RAM sistema: 79.4%


sbs_rentabilidad: 2,898 filas | 2.436 s | RAM sistema: 79.7%


sbs_valor_cuota: 76,059 filas | 2.420 s | RAM sistema: 79.8%


memorias_indicadores: 48 filas | 1.793 s | RAM sistema: 79.9%


sbs_indicadores_adicionales: 17,098 filas | 1.787 s | RAM sistema: 79.9%
tabla_analitica_mensual: 140 filas | 0.654 s | RAM sistema: 80.0%

=== RESULTADO 1X ===
Tablas procesadas: 8
Filas acumuladas: 110407
Tiempo total: 16.179 segundos


In [17]:
print("=== VERIFICACIÓN DE SALIDAS 1X ===")

total_verificado = 0

for nombre in TABLAS_BENCHMARK:

    ruta = (
        f"{RUTA_EXPERIMENTO}/escala_1x/"
        f"{nombre}"
    )

    df = spark.read.parquet(ruta)

    filas = df.count()
    replicas = df.select("replica_id").distinct().count()

    print(
        f"{nombre}: {filas:,} filas | "
        f"{replicas} réplica(s)"
    )

    assert filas == conteos[nombre]
    assert replicas == 1

    total_verificado += filas

print("\nTOTAL VERIFICADO:", f"{total_verificado:,}")

assert total_verificado == 110407

print("PRUEBA DE REPLICACIÓN 1X: CORRECTA")

=== VERIFICACIÓN DE SALIDAS 1X ===
bcrp_series_largas: 12,317 filas | 1 réplica(s)
bcrp_json_rentabilidad: 699 filas | 1 réplica(s)
sbs_afiliados: 1,148 filas | 1 réplica(s)
sbs_rentabilidad: 2,898 filas | 1 réplica(s)
sbs_valor_cuota: 76,059 filas | 1 réplica(s)
memorias_indicadores: 48 filas | 1 réplica(s)
sbs_indicadores_adicionales: 17,098 filas | 1 réplica(s)
tabla_analitica_mensual: 140 filas | 1 réplica(s)

TOTAL VERIFICADO: 110,407
PRUEBA DE REPLICACIÓN 1X: CORRECTA


In [18]:
print("=== PREPARACION DEL PIPELINE EXPERIMENTAL ===")

RUTA_1X = f"{RUTA_EXPERIMENTO}/escala_1x"

def leer_replica(nombre, escala=1):
    return spark.read.parquet(
        f"{RUTA_EXPERIMENTO}/escala_{escala}x/{nombre}"
    )

cuotas_1x = leer_replica("sbs_valor_cuota")
mensual_1x = leer_replica("tabla_analitica_mensual")

print("Valor cuota:", cuotas_1x.count())
print("Tabla mensual:", mensual_1x.count())

print("\nColumnas valor cuota:")
print(cuotas_1x.columns)

print("\nColumnas tabla mensual:")
print(mensual_1x.columns)

print("\nIdentificadores de replica:")

cuotas_1x.groupBy("replica_id").count().show()
mensual_1x.groupBy("replica_id").count().show()

=== PREPARACION DEL PIPELINE EXPERIMENTAL ===
Valor cuota: 76059
Tabla mensual: 140

Columnas valor cuota:
['fecha', 'fondo', 'afp', 'valor_cuota', 'fecha_mes', 'anio', 'replica_id']

Columnas tabla mensual:
['fecha_mes', 'rent_real_sistema_12m', 'rent_real_habitat_12m', 'rent_real_integra_12m', 'rent_real_prima_12m', 'rent_real_profuturo_12m', 'fuente_sistema', 'ipc_var_mensual_pct', 'tipo_cambio_soles_usd', 'afiliados_integra', 'valor_cuota_fin_mes_integra_f2', 'prom_otras3', 'brecha_real_pp', 'brecha_lag1', 'brecha_lag3', 'sistema_lag1', 'ipc_lag1', 'tipo_cambio_lag1', 'var_tc_12m_pct_lag1', 'anio_ejercicio', 'caracteres_memoria', 'disponible_desde', 'n_renta_variable', 'n_exterior', 'n_volatilidad', 'n_retiros', 'n_pandemia_covid', 'n_inflacion', 'n_tipo_cambio', 'n_tasa_interes', 'n_alternativos', 'n_sostenibilidad_esg', 'n_traspasos', 'n_comisiones', 'freq_renta_variable', 'freq_exterior', 'freq_volatilidad', 'freq_retiros', 'freq_pandemia_covid', 'freq_inflacion', 'freq_tipo_cam

In [19]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

def procesar_integracion(cuotas, mensual):

    cuotas_limpias = cuotas.filter(
        F.col("fecha_mes").isNotNull()
        & F.col("afp").isNotNull()
        & F.col("fondo").isNotNull()
        & F.col("valor_cuota").isNotNull()
        & (F.col("valor_cuota") > 0)
    )

    mensual_limpia = mensual.filter(
        F.col("fecha_mes").isNotNull()
        & F.col("rent_real_integra_12m").isNotNull()
    )

    integrados = cuotas_limpias.join(
        broadcast(
            mensual_limpia.select(
                "replica_id",
                "fecha_mes",
                "rent_real_integra_12m"
            )
        ),
        on=["replica_id", "fecha_mes"],
        how="inner"
    )

    resultado = (
        integrados
        .withColumn(
            "valor_ponderado",
            F.col("valor_cuota")
            * F.col("rent_real_integra_12m")
        )
        .groupBy("replica_id", "afp", "fondo")
        .agg(
            F.count("*").alias("registros"),
            F.sum("valor_ponderado").alias("total_ponderado"),
            F.avg("valor_cuota").alias("valor_cuota_promedio")
        )
    )

    return resultado

In [20]:
import time
import psutil

print("=== PROCESAMIENTO EXPERIMENTAL 1X ===")

inicio = time.perf_counter()

resultado_1x = procesar_integracion(
    cuotas_1x,
    mensual_1x
)

ruta_salida_1x = (
    f"{RUTA_EXPERIMENTO}/"
    "resultados_procesamiento/escala_1x"
)

(
    resultado_1x.write
    .mode("errorifexists")
    .parquet(ruta_salida_1x)
)

tiempo_procesamiento_1x = time.perf_counter() - inicio

guardado_1x = spark.read.parquet(ruta_salida_1x)

print("=== RESULTADOS ===")

print(
    "Tiempo de procesamiento:",
    round(tiempo_procesamiento_1x, 3),
    "segundos"
)

print("Grupos generados:", guardado_1x.count())

print(
    "Registros integrados:",
    guardado_1x.agg(
        F.sum("registros").alias("total")
    ).first()["total"]
)

print(
    "RAM del sistema:",
    psutil.virtual_memory().percent,
    "%"
)

print(
    "Swap utilizada:",
    round(
        psutil.swap_memory().used / (1024 ** 2),
        2
    ),
    "MiB"
)

guardado_1x.orderBy(
    "afp", "fondo"
).show(20, truncate=False)

=== PROCESAMIENTO EXPERIMENTAL 1X ===


=== RESULTADOS ===
Tiempo de procesamiento: 3.044 segundos
Grupos generados: 16
Registros integrados: 47112
RAM del sistema: 82.2 %
Swap utilizada: 98.57 MiB
+----------+---------+-----+---------+------------------+--------------------+
|replica_id|afp      |fondo|registros|total_ponderado   |valor_cuota_promedio|
+----------+---------+-----+---------+------------------+--------------------+
|0         |Habitat  |0    |2703     |134499.82375705734|12.669059511949662  |
|0         |Habitat  |1    |3025     |211546.31212090235|17.250632438677677  |
|0         |Habitat  |2    |3025     |224349.09426711785|18.202230887338864  |
|0         |Habitat  |3    |3025     |223716.67554761993|17.615874991801636  |
|0         |Integra  |0    |2703     |132048.1800146334 |12.427923850351437  |
|0         |Integra  |1    |3025     |318857.8076935798 |26.33354001031401   |
|0         |Integra  |2    |3025     |2452838.652222151 |201.71838736426434  |
|0         |Integra  |3    |3025     |550472.3834334

In [22]:
import time
import psutil
import json
import threading
from datetime import datetime

def ejecutar_escala(escala):

    print("=" * 60)
    print(f"PRUEBA DE ESCALABILIDAD {escala}X")
    print("=" * 60)

    ruta_escala = (
        f"{RUTA_EXPERIMENTO}/prueba_pipeline/"
        f"escala_{escala}x"
    )

    metricas = []
    muestras_recursos = []
    detener_monitoreo = threading.Event()

    def monitorear():
        while not detener_monitoreo.wait(0.5):
            ram = psutil.virtual_memory()
            swap = psutil.swap_memory()

            muestras_recursos.append({
                "ram_pct": ram.percent,
                "ram_disponible_mib":
                    ram.available / (1024 ** 2),
                "swap_usada_mib":
                    swap.used / (1024 ** 2)
            })

    monitor = threading.Thread(
        target=monitorear,
        daemon=True
    )

    inicio_total = time.perf_counter()
    monitor.start()

    try:

        print("\n=== ETAPA 1: REPLICACIÓN ===")

        for nombre in TABLAS_BENCHMARK:

            inicio = time.perf_counter()

            df = replicar_tabla(nombre, escala)

            destino = (
                f"{ruta_escala}/entrada/{nombre}"
            )

            (
                df.write
                .mode("errorifexists")
                .parquet(destino)
            )

            segundos = time.perf_counter() - inicio

            metricas.append({
                "etapa": "replicacion",
                "tabla": nombre,
                "segundos": round(segundos, 3)
            })

            print(
                f"{nombre}: {segundos:.3f} segundos"
            )


        print("\n=== ETAPA 2: VALIDACIÓN ===")

        inicio = time.perf_counter()

        total_verificado = 0

        for nombre in TABLAS_BENCHMARK:

            ruta = f"{ruta_escala}/entrada/{nombre}"

            df = spark.read.parquet(ruta)

            cantidad = df.count()
            esperado = conteos[nombre] * escala

            assert cantidad == esperado, (
                f"Cantidad incorrecta en {nombre}"
            )

            total_verificado += cantidad

        segundos_validacion = (
            time.perf_counter() - inicio
        )

        assert total_verificado == 110407 * escala

        print("Registros verificados:", total_verificado)

        metricas.append({
            "etapa": "validacion",
            "tabla": "todas",
            "segundos": round(segundos_validacion, 3)
        })


        print("\n=== ETAPA 3: INTEGRACIÓN ===")

        inicio = time.perf_counter()

        cuotas = spark.read.parquet(
            f"{ruta_escala}/entrada/sbs_valor_cuota"
        )

        mensual = spark.read.parquet(
            f"{ruta_escala}/entrada/tabla_analitica_mensual"
        )

        integrados = procesar_integracion(
            cuotas,
            mensual
        )

        destino_final = (
            f"{ruta_escala}/salida_integrada"
        )

        (
            integrados.write
            .mode("errorifexists")
            .parquet(destino_final)
        )

        segundos_integracion = (
            time.perf_counter() - inicio
        )

        resultado = spark.read.parquet(destino_final)

        grupos = resultado.count()

        registros_integrados = (
            resultado.agg(
                F.sum("registros").alias("n")
            ).first()["n"]
        )

        print("Grupos:", grupos)
        print("Registros integrados:", registros_integrados)

        assert grupos == 16 * escala
        assert registros_integrados == 47112 * escala

        metricas.append({
            "etapa": "integracion",
            "tabla": "valor_cuota_mensual",
            "segundos": round(segundos_integracion, 3)
        })

    finally:
        detener_monitoreo.set()
        monitor.join(timeout=2)

    tiempo_total = time.perf_counter() - inicio_total

    resultado_prueba = {
        "escala": escala,
        "filas_entrada": 110407 * escala,
        "filas_integradas": registros_integrados,
        "grupos_salida": grupos,
        "tiempo_total_segundos": round(tiempo_total, 3),
        "tiempo_integracion_segundos": round(
            segundos_integracion, 3
        ),
        "ram_maxima_pct": max(
            [m["ram_pct"] for m in muestras_recursos],
            default=psutil.virtual_memory().percent
        ),
        "ram_minima_disponible_mib": min(
            [m["ram_disponible_mib"]
             for m in muestras_recursos],
            default=psutil.virtual_memory().available / (1024 ** 2)
        ),
        "swap_maxima_usada_mib": max(
            [m["swap_usada_mib"]
             for m in muestras_recursos],
            default=psutil.swap_memory().used / (1024 ** 2)
        ),
        "etapas": metricas,
        "ruta": ruta_escala
    }

    print("\n=== RESUMEN FINAL ===")

    print(
        json.dumps(
            resultado_prueba,
            indent=2,
            ensure_ascii=False
        )
    )

    return resultado_prueba

In [23]:
resultado_10x = ejecutar_escala(10)

PRUEBA DE ESCALABILIDAD 10X

=== ETAPA 1: REPLICACIÓN ===


bcrp_series_largas: 1.536 segundos


bcrp_json_rentabilidad: 1.288 segundos
sbs_afiliados: 0.735 segundos


sbs_rentabilidad: 1.399 segundos


sbs_valor_cuota: 1.051 segundos
memorias_indicadores: 0.735 segundos
sbs_indicadores_adicionales: 0.824 segundos
tabla_analitica_mensual: 0.880 segundos

=== ETAPA 2: VALIDACIÓN ===
Registros verificados: 1104070

=== ETAPA 3: INTEGRACIÓN ===


Grupos: 160
Registros integrados: 471120

=== RESUMEN FINAL ===
{
  "escala": 10,
  "filas_entrada": 1104070,
  "filas_integradas": 471120,
  "grupos_salida": 160,
  "tiempo_total_segundos": 12.809,
  "tiempo_integracion_segundos": 2.014,
  "ram_maxima_pct": 82.4,
  "ram_minima_disponible_mib": 1400.515625,
  "swap_maxima_usada_mib": 99.5859375,
  "etapas": [
    {
      "etapa": "replicacion",
      "tabla": "bcrp_series_largas",
      "segundos": 1.536
    },
    {
      "etapa": "replicacion",
      "tabla": "bcrp_json_rentabilidad",
      "segundos": 1.288
    },
    {
      "etapa": "replicacion",
      "tabla": "sbs_afiliados",
      "segundos": 0.735
    },
    {
      "etapa": "replicacion",
      "tabla": "sbs_rentabilidad",
      "segundos": 1.399
    },
    {
      "etapa": "replicacion",
      "tabla": "sbs_valor_cuota",
      "segundos": 1.051
    },
    {
      "etapa": "replicacion",
      "tabla": "memorias_indicadores",
      "segundos": 0.735
    },
    {
      "etapa

In [24]:
ruta_10x = (
    EVIDENCIAS / "17_escalabilidad_10x.json"
)

with ruta_10x.open("w", encoding="utf-8") as f:
    json.dump(
        resultado_10x,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Evidencia guardada:", ruta_10x)

Evidencia guardada: /home/junior/BigData_UPAO/proyecto_afp_v2/evidencias/17_escalabilidad_10x.json


In [25]:
resultado_1x_comparable = ejecutar_escala(1)

PRUEBA DE ESCALABILIDAD 1X

=== ETAPA 1: REPLICACIÓN ===


bcrp_series_largas: 1.456 segundos


bcrp_json_rentabilidad: 1.520 segundos


sbs_afiliados: 0.997 segundos


sbs_rentabilidad: 1.662 segundos
sbs_valor_cuota: 0.904 segundos


memorias_indicadores: 0.917 segundos
sbs_indicadores_adicionales: 0.657 segundos
tabla_analitica_mensual: 0.309 segundos

=== ETAPA 2: VALIDACIÓN ===
Registros verificados: 110407

=== ETAPA 3: INTEGRACIÓN ===


Grupos: 16
Registros integrados: 47112

=== RESUMEN FINAL ===
{
  "escala": 1,
  "filas_entrada": 110407,
  "filas_integradas": 47112,
  "grupos_salida": 16,
  "tiempo_total_segundos": 12.26,
  "tiempo_integracion_segundos": 1.704,
  "ram_maxima_pct": 83.0,
  "ram_minima_disponible_mib": 1347.89453125,
  "swap_maxima_usada_mib": 100.3359375,
  "etapas": [
    {
      "etapa": "replicacion",
      "tabla": "bcrp_series_largas",
      "segundos": 1.456
    },
    {
      "etapa": "replicacion",
      "tabla": "bcrp_json_rentabilidad",
      "segundos": 1.52
    },
    {
      "etapa": "replicacion",
      "tabla": "sbs_afiliados",
      "segundos": 0.997
    },
    {
      "etapa": "replicacion",
      "tabla": "sbs_rentabilidad",
      "segundos": 1.662
    },
    {
      "etapa": "replicacion",
      "tabla": "sbs_valor_cuota",
      "segundos": 0.904
    },
    {
      "etapa": "replicacion",
      "tabla": "memorias_indicadores",
      "segundos": 0.917
    },
    {
      "etapa": "r

In [26]:
ruta_1x = EVIDENCIAS / "17_escalabilidad_1x.json"

with ruta_1x.open("w", encoding="utf-8") as f:
    json.dump(
        resultado_1x_comparable,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Evidencia guardada:", ruta_1x)

Evidencia guardada: /home/junior/BigData_UPAO/proyecto_afp_v2/evidencias/17_escalabilidad_1x.json


In [27]:
import pandas as pd

comparacion = pd.DataFrame([
    resultado_1x_comparable,
    resultado_10x
])

columnas = [
    "escala",
    "filas_entrada",
    "filas_integradas",
    "grupos_salida",
    "tiempo_total_segundos",
    "tiempo_integracion_segundos",
    "ram_maxima_pct",
    "swap_maxima_usada_mib"
]

print("=== COMPARACIÓN 1X VS 10X ===")

print(
    comparacion[columnas]
    .sort_values("escala")
    .to_string(index=False)
)

tiempo_1x = resultado_1x_comparable["tiempo_total_segundos"]
tiempo_10x = resultado_10x["tiempo_total_segundos"]

print("\nFACTOR DE CRECIMIENTO DEL TIEMPO:")
print(round(tiempo_10x / tiempo_1x, 2))

print("\nFACTOR DE CRECIMIENTO DE FILAS:")
print(10)

=== COMPARACIÓN 1X VS 10X ===
 escala  filas_entrada  filas_integradas  grupos_salida  tiempo_total_segundos  tiempo_integracion_segundos  ram_maxima_pct  swap_maxima_usada_mib
      1         110407             47112             16                 12.260                        1.704            83.0             100.335938
     10        1104070            471120            160                 12.809                        2.014            82.4              99.585938

FACTOR DE CRECIMIENTO DEL TIEMPO:
1.04

FACTOR DE CRECIMIENTO DE FILAS:
10


In [28]:
import json
import time
import traceback
from datetime import datetime

print("=== INICIO DE PRUEBA DE ESCALABILIDAD 100X ===")

inicio_100x = datetime.now()
print("Inicio:", inicio_100x.isoformat())

try:
    resultado_100x = ejecutar_escala(100)

    ruta_100x = EVIDENCIAS / "17_escalabilidad_100x.json"

    with ruta_100x.open("w", encoding="utf-8") as f:
        json.dump(
            resultado_100x,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("\n=== PRUEBA 100X FINALIZADA ===")
    print("Evidencia guardada:", ruta_100x)

except Exception as e:
    print("\n=== PRUEBA 100X NO COMPLETADA ===")
    print("Error:", repr(e))

    reporte_error = {
        "escala": 100,
        "estado": "fallida",
        "inicio": inicio_100x.isoformat(),
        "fin": datetime.now().isoformat(),
        "error": repr(e),
        "detalle": traceback.format_exc()
    }

    ruta_error = EVIDENCIAS / "17_escalabilidad_100x_error.json"

    with ruta_error.open("w", encoding="utf-8") as f:
        json.dump(
            reporte_error,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("Error registrado en:", ruta_error)

=== INICIO DE PRUEBA DE ESCALABILIDAD 100X ===
Inicio: 2026-10-11T10:49:26.306092
PRUEBA DE ESCALABILIDAD 100X

=== ETAPA 1: REPLICACIÓN ===


bcrp_series_largas: 2.970 segundos


bcrp_json_rentabilidad: 1.048 segundos


sbs_afiliados: 1.717 segundos


sbs_rentabilidad: 1.636 segundos


sbs_valor_cuota: 5.284 segundos
memorias_indicadores: 0.608 segundos


sbs_indicadores_adicionales: 2.726 segundos
tabla_analitica_mensual: 0.719 segundos

=== ETAPA 2: VALIDACIÓN ===
Registros verificados: 11040700

=== ETAPA 3: INTEGRACIÓN ===


Grupos: 1600
Registros integrados: 4711200

=== RESUMEN FINAL ===
{
  "escala": 100,
  "filas_entrada": 11040700,
  "filas_integradas": 4711200,
  "grupos_salida": 1600,
  "tiempo_total_segundos": 22.923,
  "tiempo_integracion_segundos": 3.738,
  "ram_maxima_pct": 83.6,
  "ram_minima_disponible_mib": 1303.53515625,
  "swap_maxima_usada_mib": 116.359375,
  "etapas": [
    {
      "etapa": "replicacion",
      "tabla": "bcrp_series_largas",
      "segundos": 2.97
    },
    {
      "etapa": "replicacion",
      "tabla": "bcrp_json_rentabilidad",
      "segundos": 1.048
    },
    {
      "etapa": "replicacion",
      "tabla": "sbs_afiliados",
      "segundos": 1.717
    },
    {
      "etapa": "replicacion",
      "tabla": "sbs_rentabilidad",
      "segundos": 1.636
    },
    {
      "etapa": "replicacion",
      "tabla": "sbs_valor_cuota",
      "segundos": 5.284
    },
    {
      "etapa": "replicacion",
      "tabla": "memorias_indicadores",
      "segundos": 0.608
    },
    {
     

In [29]:
import pandas as pd

resultados_escalabilidad = [
    resultado_1x_comparable,
    resultado_10x,
    resultado_100x
]

columnas = [
    "escala",
    "filas_entrada",
    "filas_integradas",
    "grupos_salida",
    "tiempo_total_segundos",
    "tiempo_integracion_segundos",
    "ram_maxima_pct",
    "ram_minima_disponible_mib",
    "swap_maxima_usada_mib"
]

tabla_resultados = pd.DataFrame(
    resultados_escalabilidad
)[columnas].sort_values("escala")

print("=== RESULTADOS FINALES DE ESCALABILIDAD ===")
print(tabla_resultados.to_string(index=False))

base = resultado_1x_comparable["tiempo_total_segundos"]

print("\n=== FACTORES DE CRECIMIENTO ===")

for r in resultados_escalabilidad:
    factor = r["tiempo_total_segundos"] / base

    print(
        f"Escala {r['escala']}x:",
        f"tiempo = {factor:.2f}x",
        f"| volumen = {r['escala']}x"
    )

destino_csv = (
    EVIDENCIAS / "17_comparacion_escalabilidad.csv"
)

tabla_resultados.to_csv(
    destino_csv,
    index=False,
    encoding="utf-8"
)

print("\nTabla guardada:", destino_csv)

=== RESULTADOS FINALES DE ESCALABILIDAD ===
 escala  filas_entrada  filas_integradas  grupos_salida  tiempo_total_segundos  tiempo_integracion_segundos  ram_maxima_pct  ram_minima_disponible_mib  swap_maxima_usada_mib
      1         110407             47112             16                 12.260                        1.704            83.0                1347.894531             100.335938
     10        1104070            471120            160                 12.809                        2.014            82.4                1400.515625              99.585938
    100       11040700           4711200           1600                 22.923                        3.738            83.6                1303.535156             116.359375

=== FACTORES DE CRECIMIENTO ===
Escala 1x: tiempo = 1.00x | volumen = 1x
Escala 10x: tiempo = 1.04x | volumen = 10x
Escala 100x: tiempo = 1.87x | volumen = 100x

Tabla guardada: /home/junior/BigData_UPAO/proyecto_afp_v2/evidencias/17_comparacion_escalabilidad